# 4) Probabilistic forecasts — Exercises

**Goals:** implement and test the pinball loss, check calibration at several levels, try to repair seasonal miscalibration, size a reserve from forecast errors, and compare two kinds of solar interval.

The setup cell holds the lecture's honest 2024 forecast errors `ERRORS_2024`, the 2025 point forecast `POINT_2025` and actuals `ACTUAL_2025` (MW), and the solar data `SOLAR` (MW, 1,500 MW fleet).

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# forecast_data.py lives in the module folder: search upwards so this works from exercise/ and exercise/solutions/
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "forecast_data.py").exists())))
from forecast_data import load_data, solar_data, temperature_forecast

warnings.filterwarnings("ignore")
df = load_data()
df["temperature_forecast"] = temperature_forecast(df)
dow = df.index.dayofweek
df["daytype"] = np.where(df.holiday | (dow == 6), "sunday_or_holiday", np.where(dow == 5, "saturday", "workday"))
df["hour"], df["month"] = df.index.hour, df.index.month
FORMULA = "demand ~ C(hour):C(daytype) + C(month) + bs(temperature, df=6, lower_bound=-12, upper_bound=40)"
honest = df.assign(temperature=df.temperature_forecast)

# the lecture's honest errors (2023 model forecasting 2024) and its 2025 point forecast (2023-2024 model)
ERRORS_2024 = (df.demand - smf.ols(FORMULA, df.loc["2023"]).fit().predict(honest)).loc["2024"]
POINT_2025 = smf.ols(FORMULA, df.loc[:"2024"]).fit().predict(honest).loc["2025"]
ACTUAL_2025 = df.demand.loc["2025"]
SOLAR = solar_data(df)

### Warm-ups

**1. The pinball loss**

Write `pinball(y, q, alpha)`. Then check its defining property on 100,000 draws from a normal distribution with mean 0 and standard deviation 1: over a grid of candidate values q from −3 to 3, the loss for α = 0.9 must be smallest near the true 90 % quantile, 1.2816.

In [ ]:
def pinball(y, q, alpha: float) -> float:
    """Mean pinball loss of the alpha-quantile forecast q."""
    ...  # YOUR CODE HERE

In [ ]:
draws = np.random.default_rng(0).normal(0, 1, 100_000)
grid = np.round(np.arange(-3, 3.001, 0.01), 2)
losses = pd.Series([pinball(draws, q, 0.9) for q in grid], index=grid)
assert abs(losses.idxmin() - 1.2816) < 0.03, "the pinball loss is minimised by the true quantile"
assert np.isclose(pinball([10.0], [8.0], 0.9), 0.9 * 2) and np.isclose(pinball([10.0], [12.0], 0.9), 0.1 * 2)
print(f"loss minimised at q = {losses.idxmin():.2f} (true 90 % quantile 1.2816)")

**2. Calibration at several levels**

Build central intervals at nominal levels 50 %, 80 %, 90 % and 95 % from the quantiles of `ERRORS_2024` around `POINT_2025`, and report the share of 2025 hours inside each. Write `reliability(levels)`.

In [ ]:
def reliability(levels=(0.5, 0.8, 0.9, 0.95)) -> pd.DataFrame:
    """Nominal level, actual 2025 coverage and width (MW) of central intervals from the 2024 errors."""
    ...  # YOUR CODE HERE

In [ ]:
rel = reliability()
assert (np.abs(rel.coverage - rel.index) < 0.05).all(), "within 5 points of nominal at every level"
assert rel.width_MW.is_monotonic_increasing
rel.round(3)

### Core

**3. Can month-specific widths repair seasonal miscalibration?**

The lecture's interval covered between 72 % and 87 % depending on the month. Build an 80 % interval from **month-specific** quantiles of `ERRORS_2024` instead of one set for the whole year. Report the spread of monthly coverage and the mean pinball loss (of the 10 % and 90 % bounds) for both versions. Write `monthly_intervals()`.

In [ ]:
def monthly_intervals() -> pd.DataFrame:
    """Overall coverage, lowest and highest monthly coverage, and mean pinball loss of two 80 % intervals."""
    ...  # YOUR CODE HERE

In [ ]:
monthly = monthly_intervals()
assert np.isclose(monthly.loc["one interval for the year", "coverage"], 0.806, atol=0.001)
monthly.round(3)

**4. Sizing a reserve**

A TSO wants enough upward reserve to cover the demand forecast's under-forecasts in 99 % of hours. Size it from `ERRORS_2024` (demand − forecast) as their 99 % quantile, then check 2025: in how many hours did demand exceed the forecast by more than the reserve, and by how much at worst? Repeat for 95 % and 99.9 %. Write `reserve_sizing(levels)`.

In [ ]:
def reserve_sizing(levels=(0.95, 0.99, 0.999)) -> pd.DataFrame:
    """Reserve (MW) sized on 2024 errors, and the 2025 hours and worst shortfall beyond it."""
    ...  # YOUR CODE HERE

In [ ]:
reserves = reserve_sizing()
assert reserves.reserve_MW.is_monotonic_increasing
assert reserves.loc[0.99, "hours_short_2025"] > 0
reserves.round(1)

**5. Interpretation**

Did month-specific quantiles fix the seasonal miscalibration — and did the pinball loss reward them? What did the reserve check reveal about sizing from one year of errors, especially at 99.9 %?

*Your answer here.*

### Challenge

**6. Solar: one width or a width that follows the forecast?**

Compare two 80 % solar intervals for 2025, both built from 2024's errors and clipped to 0–1,500 MW: (a) one constant width from the quantiles of 2024's **daylight** errors, and (b) the lecture's widths by forecast level (bins −1, 1, 100, 300, 600, 900, 1,500 MW). Score both on **daylight** hours of 2025 (actual output above 1 MW): coverage, mean width and mean pinball loss. Write `solar_intervals()`.

In [ ]:
def solar_intervals() -> pd.DataFrame:
    """Daylight coverage, mean width (MW) and mean pinball loss of two 80 % solar intervals for 2025."""
    ...  # YOUR CODE HERE

In [ ]:
solar_table = solar_intervals()
assert solar_table.loc["width by forecast level", "pinball_mean"] < solar_table.loc["constant width", "pinball_mean"]
solar_table.round(3)

**7. Interpretation**

Which solar interval is better, and how do coverage, width and pinball loss each tell the story? Why does the constant-width interval look acceptable on coverage alone?

*Your answer here.*